# PromptGFM-Bio — Kaggle Training Notebook
**Gene-Phenotype Prediction for Rare Diseases**

### ✅ Resumable Across Sessions & Accounts
This notebook saves **three Kaggle Datasets** after training so any future session — or a different Kaggle account — can skip all expensive steps:

| Dataset name (you choose) | What it stores | Skips |
|---|---|---|
| `promptgfm-model-cache` | HuggingFace BioBERT weights | ~5 min download |
| `promptgfm-data` | Raw + processed graph | ~25 min download + preprocess |
| `promptgfm-checkpoints` | Per-epoch checkpoints | training from epoch 0 |

**Setup once → add as Dataset inputs on every future session.**

> ⚙️ Before running: Settings → Accelerator → **GPU T4 x2** · Internet → **On**

## 1. Environment Check

In [1]:
import sys, subprocess, os
import torch

print(f'Python  : {sys.version}')
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.version.cuda}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'VRAM    : {vram:.1f} GB  (expect ~15-16 GB on T4)')
else:
    print('⚠️  No GPU — enable in Notebook Settings → Accelerator → GPU T4')


Python  : 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch : 2.9.0+cu126
CUDA    : 12.6
GPU     : Tesla T4
VRAM    : 15.6 GB  (expect ~15-16 GB on T4)


## 2. ⚙️ Session Configuration
Edit the variables below **before running any other cell**.

**`RESUME_*` flags**: set to `True` if you have added the corresponding Kaggle Dataset as input.  
**Dataset input paths**: change if you named your datasets differently.

In [2]:
# ─── RESUME FLAGS ────────────────────────────────────────────────────────
# Set True when you have added the matching dataset as notebook input
RESUME_HF_CACHE     = False  # True → skip BioBERT download (saves ~5 min)
RESUME_DATA         = False  # True → skip raw download + preprocessing (~25 min)
RESUME_CHECKPOINTS  = False  # True → resume training from last saved epoch

# ─── INPUT DATASET PATHS (Kaggle mounts datasets under /kaggle/input/) ───
# After you create and add them, the paths will match these names:
INPUT_HF_CACHE    = '/kaggle/input/promptgfm-model-cache'
INPUT_DATA        = '/kaggle/input/promptgfm-data'
INPUT_CHECKPOINTS = '/kaggle/input/promptgfm-checkpoints'

# ─── OUTPUT DATASET NAMES (used in Step 12 instructions) ─────────────────
OUTPUT_HF_CACHE    = 'promptgfm-model-cache'
OUTPUT_DATA        = 'promptgfm-data'
OUTPUT_CHECKPOINTS = 'promptgfm-checkpoints'

# ─── HF MODEL CACHE LOCATION ─────────────────────────────────────────────
# Point HuggingFace to a path inside /kaggle/working/ so we can save it
HF_CACHE_DIR = '/kaggle/working/hf_cache'
os.environ['HF_HOME']              = HF_CACHE_DIR
os.environ['TRANSFORMERS_CACHE']   = HF_CACHE_DIR
os.environ['HUGGINGFACE_HUB_CACHE']= HF_CACHE_DIR

print('Configuration:')
print(f'  RESUME_HF_CACHE    = {RESUME_HF_CACHE}')
print(f'  RESUME_DATA        = {RESUME_DATA}')
print(f'  RESUME_CHECKPOINTS = {RESUME_CHECKPOINTS}')
print(f'  HF cache dir       = {HF_CACHE_DIR}')


Configuration:
  RESUME_HF_CACHE    = False
  RESUME_DATA        = False
  RESUME_CHECKPOINTS = False
  HF cache dir       = /kaggle/working/hf_cache


## 3. Install PyTorch Geometric

In [3]:
import torch, subprocess, sys

TORCH_VER = torch.__version__.split('+')[0]
CUDA_VER  = torch.version.cuda.replace('.', '')
WHEEL_URL = f'https://data.pyg.org/whl/torch-{TORCH_VER}+cu{CUDA_VER}.html'
print(f'PyG wheel source: {WHEEL_URL}')

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--quiet',
     '-f', WHEEL_URL,
     'torch-scatter', 'torch-sparse', 'torch-cluster',
     'torch-spline-conv', 'torch-geometric'],
    check=True
)
print('✅ PyTorch Geometric installed')


PyG wheel source: https://data.pyg.org/whl/torch-2.9.0+cu126.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 62.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 66.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 96.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 61.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.2 MB/s eta 0:00:00
✅ PyTorch Geometric installed


## 4. Install Extra Dependencies

In [4]:
# Upgrade build tools first — prevents metadata-generation-failed on Python 3.12
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '--upgrade', 'setuptools', 'wheel', 'pip'], check=True)

extra = [
    'transformers>=4.40.0',
    'sentence-transformers>=2.7.0',
    'biopython>=1.84',
    'networkx>=3.2',
    'wandb>=0.17.0',
    'python-dotenv>=1.0.0',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet'] + extra, check=True)
print('✅ Extra packages installed')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 62.7 MB/s eta 0:00:00
✅ Extra packages installed


## 5. Clone Project Code from GitHub

In [5]:
import os
from pathlib import Path

GITHUB_URL  = 'https://github.com/pes1ug23am910/PROMPTGMF-Bio-Kaggle.git'
PROJECT_DIR = '/kaggle/working/PromptGFM-Bio'

if not os.path.exists(PROJECT_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', GITHUB_URL, PROJECT_DIR], check=True)
    print(f'✅ Cloned to {PROJECT_DIR}')
else:
    subprocess.run(['git', '-C', PROJECT_DIR, 'pull'], check=True)
    print(f'✅ Pulled latest changes')

os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')


Cloning into '/kaggle/working/PromptGFM-Bio'...


✅ Cloned to /kaggle/working/PromptGFM-Bio
Working directory: /kaggle/working/PromptGFM-Bio


## 6. Create Directory Structure

In [6]:
from pathlib import Path

dirs = [
    'data/raw', 'data/processed', 'data/splits',
    'checkpoints/promptgfm_film',
    'logs',
]
for d in dirs:
    Path(d).mkdir(parents=True, exist_ok=True)
print('✅ Directories created')


✅ Directories created


## 7. Restore Assets from Previous Session
Restores HuggingFace model cache, preprocessed data, and training checkpoints from saved Kaggle Datasets — skipping all expensive steps below.
**First-time run**: all three blocks will print "not found — will download/preprocess fresh".

In [7]:
import shutil, tarfile
from pathlib import Path

def restore_tar(src_path, dest_dir, label):
    """Extract a .tar.gz archive if it exists."""
    src = Path(src_path)
    if src.exists():
        dest = Path(dest_dir)
        dest.mkdir(parents=True, exist_ok=True)
        with tarfile.open(src, 'r:gz') as tar:
            tar.extractall(dest)
        print(f'✅ {label} restored from {src}')
        return True
    return False

def restore_dir(src_path, dest_dir, label):
    """Copy directory tree if source exists."""
    src = Path(src_path)
    if src.exists() and any(src.iterdir()):
        dest = Path(dest_dir)
        if dest.exists():
            shutil.rmtree(dest)
        shutil.copytree(src, dest)
        print(f'✅ {label} restored from {src}')
        return True
    return False

# ── A. HuggingFace model cache ────────────────────────────────────────────
if RESUME_HF_CACHE:
    ok = restore_tar(f'{INPUT_HF_CACHE}/hf_cache.tar.gz', HF_CACHE_DIR, 'HF model cache')
    if not ok:
        ok = restore_dir(f'{INPUT_HF_CACHE}/hf_cache', HF_CACHE_DIR, 'HF model cache')
    if not ok:
        print(f'⚠️  HF cache not found at {INPUT_HF_CACHE} — BioBERT will be re-downloaded')
else:
    print('HF cache: skipped (RESUME_HF_CACHE=False)')

# ── B. Preprocessed graph + raw data ─────────────────────────────────────
if RESUME_DATA:
    ok = restore_tar(f'{INPUT_DATA}/data.tar.gz', 'data', 'Graph data')
    if not ok:
        ok = restore_dir(f'{INPUT_DATA}/processed', 'data/processed', 'Processed graph')
        restore_dir(f'{INPUT_DATA}/splits', 'data/splits', 'Data splits')
    graph = Path('data/processed/biomedical_graph.pt')
    if graph.exists():
        print(f'✅ Graph ready ({graph.stat().st_size/1e6:.0f} MB)')
    else:
        print('⚠️  Graph not found in restored data — will preprocess fresh')
        RESUME_DATA = False   # force re-preprocessing below
else:
    print('Data: skipped (RESUME_DATA=False)')

# ── C. Training checkpoints ───────────────────────────────────────────────
if RESUME_CHECKPOINTS:
    ckpt_src = Path(INPUT_CHECKPOINTS)
    ckpt_dst = Path('checkpoints/promptgfm_film')
    ckpt_dst.mkdir(parents=True, exist_ok=True)
    files_copied = 0
    if ckpt_src.exists():
        for f in ckpt_src.glob('*'):
            shutil.copy2(f, ckpt_dst / f.name)
            files_copied += 1
    if files_copied:
        epochs = sorted([f.stem.replace('checkpoint_epoch_','') for f in ckpt_dst.glob('checkpoint_epoch_*.pt')])
        print(f'✅ Checkpoints restored ({files_copied} files). Epochs available: {epochs}')
    else:
        print(f'⚠️  No checkpoints found at {INPUT_CHECKPOINTS} — will train from scratch')
        RESUME_CHECKPOINTS = False
else:
    print('Checkpoints: skipped (RESUME_CHECKPOINTS=False)')


HF cache: skipped (RESUME_HF_CACHE=False)
Data: skipped (RESUME_DATA=False)
Checkpoints: skipped (RESUME_CHECKPOINTS=False)


## 7.5. Pre-download BioBERT Model
Downloads the full **440 MB** PubMedBERT weights to disk **before** training.

**Why this cell exists:** The model uses safetensors lazy-loading, which streams weights from the network one parameter at a time. If Kaggle's connection to HuggingFace CDN hiccups mid-stream (~31% is a common stall point), the training cell hangs indefinitely.

Running `snapshot_download` here downloads the complete file to disk first.  
**If this cell stalls:** click ■ Stop → re-run only this cell — it resumes from where it left off.

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path
import warnings, os

MODEL_NAME = 'microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext'

hub_dir = Path(HF_CACHE_DIR) / 'hub'
model_cache_name = 'models--' + MODEL_NAME.replace('/', '--')
model_snapshots = hub_dir / model_cache_name / 'snapshots'

if RESUME_HF_CACHE and model_snapshots.exists() and any(model_snapshots.iterdir()):
    print('⏭️  BioBERT already in restored HF cache — skipping download')
else:
    print(f'Downloading {MODEL_NAME}')
    print(f'  Cache dir : {HF_CACHE_DIR}')
    print(f'  Size      : ~440 MB — takes ~2-3 min on Kaggle')
    print(f'  Downloads resume automatically if interrupted\n')
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', UserWarning)
        snapshot_download(
            repo_id=MODEL_NAME,
            cache_dir=HF_CACHE_DIR,
            ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*', 'rust_model*', '*.ot'],
        )
    print('\n✅ BioBERT fully downloaded and cached')

# ── Force offline mode so training NEVER touches the network for model weights ──
# This prevents from_pretrained() making freshness-check HEAD requests that can
# stall indefinitely on Kaggle's flaky HuggingFace CDN connection.
os.environ['TRANSFORMERS_OFFLINE'] = '1'
os.environ['HF_HUB_OFFLINE']       = '1'
print('✅ Offline mode enabled — training will load BioBERT from disk only')


  Cache dir : /kaggle/working/hf_cache
  Size      : ~440 MB — takes ~2-3 min on Kaggle
  Resume    : yes (safe to re-run if this stalls)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:186: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(


Fetching 7 files:   0%|          | 0/7 [00:00<?, ?it/s]


✅ BioBERT fully downloaded and cached — training will load from disk


## 8. Download Biomedical Datasets
Skipped automatically if `RESUME_DATA=True` and graph was restored successfully.  
Total download: ~1.5 GB · takes ~10 min.

In [9]:
from pathlib import Path

graph_exists = Path('data/processed/biomedical_graph.pt').exists()

if RESUME_DATA and graph_exists:
    print('⏭️  Download skipped — restored from Kaggle Dataset')
else:
    print('Downloading datasets...')
    result = subprocess.run(
        [sys.executable, 'scripts/download_data.py', '--dataset', 'all'],
        capture_output=False
    )
    print('Download exit code:', result.returncode)


INFO:src.data.download:======================================================================
INFO:src.data.download:Starting download of all biomedical datasets...
INFO:src.data.download:This may take 30-60 minutes depending on your connection
INFO:src.data.download:Total size: ~1.5 GB
INFO:src.data.download:======================================================================
INFO:src.data.download:
[1/4] BioGRID Protein-Protein Interactions
INFO:src.data.download:Downloading BioGRID database (~500MB)...
INFO:src.data.download:Downloading from https://downloads.thebiogrid.org/Download/BioGRID/Release-Archive/BIOGRID-4.4.224/BIOGRID-ALL-4.4.224.tab3.zip (attempt 1/3)



PromptGFM-Bio Data Download

Dataset: all
Force re-download: False
Skip failing: True



BIOGRID-ALL-4.4.224.tab3.zip: 160MB [00:02, 70.2MB/s] 
INFO:src.data.download:Successfully downloaded to /kaggle/working/PromptGFM-Bio/data/raw/biogrid/BIOGRID-ALL-4.4.224.tab3.zip
INFO:src.data.download:File BIOGRID-ALL-4.4.224.tab3.zip md5: ed6819e6e65ec7e685e85a0d66060ba9
INFO:src.data.download:Extracting BIOGRID-ALL-4.4.224.tab3.zip...
INFO:src.data.download:Extraction complete!
INFO:src.data.download:
[2/4] STRING Protein Network
INFO:src.data.download:Downloading STRING database for organism 9606 (~700MB)...
INFO:src.data.download:Downloading from https://stringdb-downloads.org/download/protein.links.v12.0/9606.protein.links.v12.0.txt.gz (attempt 1/3)
9606.protein.links.v12.0.txt.gz: 100%|██████████| 83.2M/83.2M [00:03<00:00, 24.6MB/s]
INFO:src.data.download:Successfully downloaded to /kaggle/working/PromptGFM-Bio/data/raw/string/9606.protein.links.v12.0.txt.gz
INFO:src.data.download:File 9606.protein.links.v12.0.txt.gz md5: 5b7c91d02926e723ad30e81ab9d009a4
INFO:src.data.download


✓ Successfully downloaded 4/4 datasets

✓ DOWNLOAD COMPLETE

Next steps:
  1. Run preprocessing: python scripts/preprocess_all.py
  2. Check downloaded files in: data/raw/

Download exit code: 0


INFO:src.data.download:File phenotype.hpoa md5: b249dc767256e904d1e92895184009b2
INFO:src.data.download:
INFO:src.data.download:Download Summary:
INFO:src.data.download:✓ BIOGRID: 2 files downloaded
INFO:src.data.download:✓ STRING: 3 files downloaded
INFO:src.data.download:✓ DISGENET: 2 files downloaded
INFO:src.data.download:✓ HPO: 4 files downloaded
INFO:src.data.download:======================================================================


## 9. Preprocess Data (Build Knowledge Graph)
Skipped automatically if `RESUME_DATA=True` and graph was restored successfully.

In [10]:
from pathlib import Path

graph_path = Path('data/processed/biomedical_graph.pt')

if RESUME_DATA and graph_path.exists():
    print(f'⏭️  Preprocessing skipped — graph ready ({graph_path.stat().st_size/1e6:.0f} MB)')
else:
    print('Building knowledge graph...')
    result = subprocess.run(
        [sys.executable, 'scripts/preprocess_all.py'],
        capture_output=False
    )
    print('Preprocessing exit code:', result.returncode)
    if graph_path.exists():
        print(f'✅ Graph ready ({graph_path.stat().st_size/1e6:.0f} MB)')
    else:
        raise RuntimeError('Graph file not created — check logs above')


Building knowledge graph...


INFO:src.data.preprocess:======================================================================
INFO:src.data.preprocess:Starting enhanced preprocessing pipeline...
INFO:src.data.preprocess:======================================================================
INFO:src.data.preprocess:Options:
INFO:src.data.preprocess:  HPO Bridge: True
INFO:src.data.preprocess:  Orphadata: True
INFO:src.data.preprocess:  UniProt: False
INFO:src.data.preprocess:  Pathways: False
INFO:src.data.preprocess:======================================================================
INFO:src.data.preprocess:
[Step 1] Parsing PPI networks...
INFO:src.data.preprocess:Parsing BioGRID from /kaggle/working/PromptGFM-Bio/data/raw/biogrid/BIOGRID-ALL-4.4.224.tab3.txt...
ERROR:src.data.preprocess:Failed to parse BioGRID: Usecols do not match columns, columns expected but not found: ['Organism Interactor A', 'Organism Interactor B']
INFO:src.data.preprocess:Parsing STRING from /kaggle/working/PromptGFM-Bio/data/raw/strin


PromptGFM-Bio Enhanced Preprocessing Pipeline

Configuration:
  Force reprocess: False
  HPO Bridge: True
  Orphadata: True
  UniProt: False
  Pathways: False


✓ PREPROCESSING COMPLETE

Next steps:
  1. Create dataset splits: python -m src.data.dataset
  2. Check graph file: data/processed/biomedical_graph.pt
  3. View statistics: data/processed/biomedical_graph_stats.txt

Preprocessing exit code: 0
✅ Graph ready (313 MB)


## 10. W&B Login (Optional)

In [11]:
#WANDB_API_KEY = ''   # paste your key here, or leave empty to disable
from kaggle_secrets import UserSecretsClient
WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")

if WANDB_API_KEY:
    import wandb
    wandb.login(key=WANDB_API_KEY)
    print('✅ W&B logged in')
else:
    os.environ['WANDB_MODE'] = 'disabled'
    print('W&B disabled — metrics logged to stdout only')


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: pes1ug23am910 (pes1ug23am910-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B logged in


## 11. Train
Uses `configs/kaggle_config.yaml` (T4-tuned: batch_size=64, hidden_dim=512).  
Set `RESUME=True` to continue from the last restored checkpoint.

In [ ]:
import torch as _torch
from pathlib import Path as _Path

# --- Auto-detect in-session checkpoints for same-session restart ---
_ckpt_dir = _Path("checkpoints/promptgfm_film")
_ckpts = sorted(
    _ckpt_dir.glob("checkpoint_epoch_*.pt"),
    key=lambda p: int(p.stem.split("_")[-1])
) if _ckpt_dir.exists() else []

# RESUME_CHECKPOINTS=True  = resuming from a Kaggle Dataset (next session)
# _ckpts non-empty         = in-session restart (same kernel, stopped+restarted)
RESUME = RESUME_CHECKPOINTS or bool(_ckpts)

# Always use train.py  (resume_training.py is interactive and hangs in notebooks)
base_args = ["scripts/train.py", "--config", "configs/kaggle_config.yaml"]
if RESUME and _ckpts:
    _last_ckpt = str(_ckpts[-1])
    base_args += ["--resume-checkpoint", _last_ckpt]
    print(f"In-session resume from: {_last_ckpt}")
elif RESUME:
    print("RESUME_CHECKPOINTS=True but no local checkpoints found -- starting fresh")
    RESUME = False

# --- Multi-GPU: use torchrun when 2+ GPUs are available ---
n_gpus = _torch.cuda.device_count()
print(f"GPUs available: {n_gpus}")

if n_gpus > 1:
    import shutil as _shutil
    _torchrun = _shutil.which("torchrun") or "torchrun"
    cmd = [_torchrun, f"--nproc_per_node={n_gpus}", "--master_port=29500"] + base_args
    print(f"Launching DDP on {n_gpus} x T4")
else:
    cmd = [sys.executable] + base_args
    print("Single-GPU launch")

print("Running:", " ".join(cmd))
result = subprocess.run(cmd, capture_output=False)
print("Training exit code:", result.returncode)


Running: /usr/bin/python3 scripts/train.py --config configs/kaggle_config.yaml


INFO:__main__:✓ cuDNN autotuning enabled (first epoch may be slightly slower)
INFO:__main__:Mode: finetune
INFO:__main__:Config: configs/kaggle_config.yaml
INFO:__main__:Device: cuda
INFO:__main__:✓ Mixed precision (AMP) enabled (1.5-2× speedup expected)
INFO:__main__:
INFO:__main__:Starting Supervised Fine-tuning
INFO:__main__:============================================================
INFO:__main__:Creating dataloaders...
INFO:src.data.dataset:Loading graph from data/processed/biomedical_graph.pt
INFO:src.data.dataset:Graph loaded: gene=5363, disease=16841, phenotype=11794, ('gene', 'associated_with', 'disease')=9741610, ('disease', 'rev_associated_with', 'gene')=9741610
INFO:src.data.dataset:Loading gene-disease edges from data/processed/hpo_gene_disease_edges.csv
INFO:src.data.dataset:Vocabulary: 5251 genes, 12714 diseases
INFO:src.data.dataset:Loaded 1170143 edges ({'HPO_phenotype_bridge': 1170143})
INFO:src.data.dataset:Split sizes: train=936114, val=117014, test=117015
INFO:__m

## 12. 💾 Save ALL Assets as Kaggle Output

Run this cell **before the session ends** (set a reminder before the 9-hour limit).

It saves three directories under `/kaggle/working/`:

| Directory | Contents | Create Dataset named |
|---|---|---|
| `out_model_cache/` | BioBERT weights (~440 MB) | `promptgfm-model-cache` |
| `out_data/` | Raw + processed graph (~600 MB) | `promptgfm-data` |
| `out_checkpoints/` | Per-epoch `.pt` files | `promptgfm-checkpoints` |

### After this cell completes:
1. Click **Output** tab (right panel) → you'll see these three folders
2. For **each** folder → click the ⊕ icon → **New Dataset** → use the names above
3. Make the datasets **Public** (or **Private** if you want them only for yourself)
4. Next session: **Add Data** → **Your Datasets** → add all three → set `RESUME_*=True`

### Using from a different Kaggle account:
Make the datasets **Public**, then the other account can find them by searching  
`pes1ug23am910/promptgfm-model-cache` etc. in **Add Data**.

In [ ]:
import shutil, tarfile, os
from pathlib import Path

def make_tar(src_dir, out_file, label):
    src = Path(src_dir)
    if not src.exists() or not any(src.rglob('*')):
        print(f'⚠️  {label}: source empty or missing ({src}) — skipped')
        return
    out = Path(out_file)
    out.parent.mkdir(parents=True, exist_ok=True)
    with tarfile.open(out, 'w:gz') as tar:
        tar.add(src, arcname=src.name)
    size_mb = out.stat().st_size / 1e6
    print(f'✅ {label} → {out}  ({size_mb:.0f} MB)')

def copy_dir(src_dir, out_dir, label):
    src = Path(src_dir)
    out = Path(out_dir)
    if not src.exists():
        print(f'⚠️  {label}: missing ({src}) — skipped')
        return
    if out.exists():
        shutil.rmtree(out)
    shutil.copytree(src, out)
    files = list(out.rglob('*.*'))
    total_mb = sum(f.stat().st_size for f in files) / 1e6
    print(f'✅ {label} → {out}  ({len(files)} files, {total_mb:.0f} MB)')

print('=== Saving assets for next session ===')

# A. HuggingFace model cache (BioBERT) ─────────────────────────────────────
make_tar(HF_CACHE_DIR,
         '/kaggle/working/out_model_cache/hf_cache.tar.gz',
         'HF model cache')

# B. Data (raw + processed graph + splits) ─────────────────────────────────
make_tar('data/processed',
         '/kaggle/working/out_data/data.tar.gz',
         'Processed graph data')

# C. Training checkpoints ──────────────────────────────────────────────────
copy_dir('checkpoints/promptgfm_film',
         '/kaggle/working/out_checkpoints',
         'Training checkpoints')

print()
print('=== Output summary ===')
for d in ['out_model_cache', 'out_data', 'out_checkpoints']:
    p = Path('/kaggle/working') / d
    if p.exists():
        files = list(p.rglob('*.*'))
        total_mb = sum(f.stat().st_size for f in files) / 1e6
        print(f'  /kaggle/working/{d}/  →  {len(files)} files, {total_mb:.0f} MB')

print()
print('Next steps:')
print('  1. Output tab → create 3 datasets from out_model_cache, out_data, out_checkpoints')
print('  2. Next session: add those datasets as input, set RESUME_*=True in cell 2')


## 13. Quick Evaluation

In [ ]:
from pathlib import Path

best = Path('checkpoints/promptgfm_film/best_model.pt')
if not best.exists():
    print('No best_model.pt yet — run more training epochs first')
else:
    result = subprocess.run(
        [sys.executable, 'scripts/evaluate.py',
         '--config', 'configs/kaggle_config.yaml',
         '--checkpoint', str(best)],
        capture_output=False
    )
    print('Evaluation exit code:', result.returncode)
